# Customer Churn Prediction

Projeto de classificação para identificar clientes com risco de cancelamento usando o dataset **Telco Customer Churn**.

O notebook compara Logistic Regression e Decision Tree. A Logistic Regression é mantida como modelo final, sem alteração de threshold ou tuning.


## 1. Bibliotecas

As dependências são mantidas enxutas e estão registradas em `requirements.txt`.


In [1]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
)
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier


## 2. Carregamento dos dados

O caminho é relativo ao projeto. Assim, o notebook funciona tanto quando o kernel é iniciado na raiz do repositório quanto dentro da pasta `notebooks`.


In [2]:
DATA_FILE = Path("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

if not DATA_FILE.exists():
    DATA_FILE = Path("data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

dados = pd.read_csv(DATA_FILE)

print(f"Dimensões do dataset: {dados.shape[0]} linhas e {dados.shape[1]} colunas")
dados["Churn"].value_counts().rename_axis("Churn").to_frame("quantidade")


Dimensões do dataset: 7043 linhas e 21 colunas


,quantidade
Churn,
No,5174
Yes,1869


## 3. Separação entre treino e teste

`customerID` é removida por ser apenas um identificador. Mantemos `stratify=y` para preservar aproximadamente a proporção de churn nos conjuntos de treino e teste.


In [3]:
y = dados["Churn"]
X = dados.drop(columns=["Churn", "customerID"])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print(f"Treino: {X_train.shape[0]} clientes")
print(f"Teste: {X_test.shape[0]} clientes")


Treino: 5634 clientes
Teste: 1409 clientes


## 4. Pipeline de pré-processamento e modelo final

`TotalCharges` chega como texto. O `FunctionTransformer` converte valores válidos para número e transforma valores inválidos em `NaN`. Em seguida, a mediana é aprendida somente nos dados de treino.

A pipeline completa recebe `X_train` e `X_test` crus, aplica o pré-processamento e executa a Logistic Regression.


In [4]:
def corrigir_total_charges(X):
    X = X.copy()
    X["TotalCharges"] = pd.to_numeric(X["TotalCharges"], errors="coerce")
    return X


colunas_numericas = ["tenure", "MonthlyCharges", "TotalCharges"]

colunas_binarias = [
    "gender",
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "PhoneService",
    "PaperlessBilling",
]

colunas_categoricas = [
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaymentMethod",
]

colunas_para_onehot = colunas_binarias + colunas_categoricas

tratamento_numericas = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

preprocessador = ColumnTransformer(
    transformers=[
        ("numericas", tratamento_numericas, colunas_numericas),
        (
            "categoricas",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            colunas_para_onehot,
        ),
    ]
)

conversor_total_charges = FunctionTransformer(corrigir_total_charges)

pipeline_final = Pipeline(
    [
        ("corrigir_total_charges", conversor_total_charges),
        ("preprocessador", preprocessador),
        ("modelo", LogisticRegression(max_iter=1000)),
    ]
)


## 5. Avaliação final no conjunto de teste

O modelo é ajustado somente com o conjunto de treino. Depois, recebe o conjunto de teste ainda cru para produzir as previsões finais.


In [5]:
pipeline_final.fit(X_train, y_train)
y_pred_final = pipeline_final.predict(X_test)

accuracy_final = accuracy_score(y_test, y_pred_final)
recall_final = recall_score(y_test, y_pred_final, pos_label="Yes")
precision_final = precision_score(y_test, y_pred_final, pos_label="Yes")
f1_final = f1_score(y_test, y_pred_final, pos_label="Yes")
matriz_final = confusion_matrix(y_test, y_pred_final, labels=["No", "Yes"])

print(f"Accuracy:  {accuracy_final:.10f}")
print(f"Recall:    {recall_final:.10f}")
print(f"Precision: {precision_final:.10f}")
print(f"F1:        {f1_final:.10f}")
print("\nMatriz de confusão:")
print(matriz_final)


Accuracy:  0.8055358410
Recall:    0.5588235294
Precision: 0.6572327044
F1:        0.6040462428

Matriz de confusão:
[[926 109]
 [165 209]]


## 6. Comparação dos modelos com validação cruzada

Além da Logistic Regression, avaliamos uma árvore sem limite de profundidade e uma árvore controlada com `max_depth=4`. A comparação entre accuracy de treino e validação ajuda a identificar overfitting.


In [6]:
scorer_recall = make_scorer(recall_score, pos_label="Yes")
scorer_precision = make_scorer(precision_score, pos_label="Yes")
scorer_f1 = make_scorer(f1_score, pos_label="Yes")

metricas = {
    "accuracy": "accuracy",
    "recall": scorer_recall,
    "precision": scorer_precision,
    "f1": scorer_f1,
}

pipeline_arvore = Pipeline(
    [
        ("corrigir_total_charges", conversor_total_charges),
        ("preprocessador", preprocessador),
        ("modelo", DecisionTreeClassifier(random_state=42)),
    ]
)

pipeline_arvore_controlada = Pipeline(
    [
        ("corrigir_total_charges", conversor_total_charges),
        ("preprocessador", preprocessador),
        ("modelo", DecisionTreeClassifier(max_depth=4, random_state=42)),
    ]
)

resultado_logistica = cross_validate(
    pipeline_final,
    X_train,
    y_train,
    scoring=metricas,
    cv=5,
    return_train_score=True,
)

resultado_arvore = cross_validate(
    pipeline_arvore,
    X_train,
    y_train,
    scoring=metricas,
    cv=5,
    return_train_score=True,
)

resultado_arvore_controlada = cross_validate(
    pipeline_arvore_controlada,
    X_train,
    y_train,
    scoring=metricas,
    cv=5,
    return_train_score=True,
)

comparacao_modelos = pd.DataFrame(
    {
        "Logistic Regression": {
            "accuracy_treino": resultado_logistica["train_accuracy"].mean(),
            "accuracy_validacao": resultado_logistica["test_accuracy"].mean(),
            "recall_validacao": resultado_logistica["test_recall"].mean(),
            "precision_validacao": resultado_logistica["test_precision"].mean(),
            "f1_validacao": resultado_logistica["test_f1"].mean(),
        },
        "Decision Tree": {
            "accuracy_treino": resultado_arvore["train_accuracy"].mean(),
            "accuracy_validacao": resultado_arvore["test_accuracy"].mean(),
            "recall_validacao": resultado_arvore["test_recall"].mean(),
            "precision_validacao": resultado_arvore["test_precision"].mean(),
            "f1_validacao": resultado_arvore["test_f1"].mean(),
        },
        "Decision Tree (max_depth=4)": {
            "accuracy_treino": resultado_arvore_controlada["train_accuracy"].mean(),
            "accuracy_validacao": resultado_arvore_controlada["test_accuracy"].mean(),
            "recall_validacao": resultado_arvore_controlada["test_recall"].mean(),
            "precision_validacao": resultado_arvore_controlada["test_precision"].mean(),
            "f1_validacao": resultado_arvore_controlada["test_f1"].mean(),
        },
    }
).T

comparacao_modelos.round(4)


,accuracy_treino,accuracy_validacao,recall_validacao,precision_validacao,f1_validacao
Logistic Regression,0.8056,0.8042,0.5485,0.6577,0.5980
Decision Tree,0.9984,0.7201,0.4849,0.4737,0.4789
Decision Tree (max_depth=4),0.7980,0.7872,0.4776,0.6423,0.5420


## 7. Conclusão

A árvore sem controle apresentou forte overfitting: resultado quase perfeito no treino e desempenho bem menor na validação. Limitar a profundidade reduziu essa diferença.

A **Logistic Regression** foi mantida como modelo final por apresentar resultados mais consistentes neste estudo. No teste, o Recall de aproximadamente 56% mostra que o modelo identifica pouco mais da metade dos clientes que realmente cancelam. A principal limitação são os falsos negativos, que representam clientes que cancelaram sem serem identificados pelo modelo.

O modelo não é apresentado como pronto para produção. Thresholds diferentes, novas variáveis e outros modelos são possibilidades de estudo futuro, mas não foram implementados aqui.
